In [0]:
# Check Accepted Driver Locations
display(spark.read.format("delta").load("/Volumes/proj_databricks/uber_medallion_layers/silver_accepted/silver_driver_locations"))

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7004721329278699>, line 2
      1 # Check Accepted Driver Locations
----> 2 display(spark.read.format("delta").load("/Volumes/proj_databricks/uber_medallion_layers/silver_accepted/silver_driver_locations"))

File /databricks/spark/python/pyspark/sql/readwriter.py:311, in DataFrameReader.load(self, path, format, schema, **options)
    309 self.options(**options)
    310 if isinstance(path, str):
--> 311     return self._df(self._jreader.load(path))
    312 elif path is not None:
    313     if type(path) != list:

File /databricks/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.py:1362, in JavaMember.__call__(self, *args)
   1356 command = proto.CALL_COMMAND_NAME +\
   1357     self.command_header +\
   1358     args_command +\
   1359     proto.END_COMMAND_PART
   1361 answer = self.gateway_client.send_command(com

In [0]:
# newwwwww

from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# ---------------------------------------------------------
# 1. NEW PATH CONFIGURATION
# ---------------------------------------------------------
BRONZE_VOLUME_PATH = "/Volumes/proj_databricks/uber_medallion_layers/bronze"
SILVER_VOLUME_PATH = "/Volumes/proj_databricks/uber_medallion_layers/uber_silver"

# ---------------------------------------------------------
# 2. EXACT PAYLOAD SCHEMAS
# ---------------------------------------------------------
driver_schema = StructType([
    StructField("driver_id", StringType(), True),
    StructField("city", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("timestamp", StringType(), True)
])

ride_schema = StructType([
    StructField("ride_id", StringType(), True),
    StructField("driver_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("pickup_city", StringType(), True),
    StructField("drop_city", StringType(), True),
    StructField("fare", DoubleType(), True),
    StructField("distance_km", DoubleType(), True),
    StructField("ride_status", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("timestamp", StringType(), True)
])

# ---------------------------------------------------------
# 3. READ & TRANSFORM SILVER DRIVER LOCATIONS
# ---------------------------------------------------------
df_driver_bronze = spark.readStream.format("delta").load(f"{BRONZE_VOLUME_PATH}/bronze_driver_locations")

df_driver_silver = df_driver_bronze \
    .withColumn("data", from_json(col("raw_payload"), driver_schema)) \
    .select(
        col("data.driver_id").alias("driver_id"),
        col("data.city").alias("city"),
        col("data.latitude").alias("latitude"),
        col("data.longitude").alias("longitude"),
        col("data.status").alias("driver_status"),
        to_timestamp(col("data.timestamp")).alias("event_timestamp")
    ) \
    .filter(col("driver_id").isNotNull())

# ---------------------------------------------------------
# 4. READ & TRANSFORM SILVER RIDE EVENTS
# ---------------------------------------------------------
df_ride_bronze = spark.readStream.format("delta").load(f"{BRONZE_VOLUME_PATH}/bronze_ride_events")

df_ride_silver = df_ride_bronze \
    .withColumn("data", from_json(col("raw_payload"), ride_schema)) \
    .select(
        col("data.ride_id").alias("ride_id"),
        col("data.driver_id").alias("driver_id"),
        col("data.customer_id").alias("customer_id"),
        col("data.pickup_city").alias("pickup_city"),
        col("data.drop_city").alias("drop_city"),
        col("data.fare").alias("fare_amount"),
        col("data.distance_km").alias("distance_km"),
        col("data.ride_status").alias("ride_status"),
        col("data.payment_method").alias("payment_method"),
        to_timestamp(col("data.timestamp")).alias("event_timestamp")
    ) \
    .filter(col("ride_id").isNotNull())


# 5 WRITE STREAMS TO UBER_SILVER LOCATION

q1 = df_driver_silver.writeStream.format("delta").outputMode("append") \
    .trigger(processingTime='5 seconds') \
    .option("checkpointLocation", f"{SILVER_VOLUME_PATH}/_checkpoints/silver_driver_locations") \
    .start(f"{SILVER_VOLUME_PATH}/silver_driver_locations")

q2 = df_ride_silver.writeStream.format("delta").outputMode("append") \
    .trigger(processingTime='5 seconds') \
    .option("checkpointLocation", f"{SILVER_VOLUME_PATH}/_checkpoints/silver_ride_events") \
    .start(f"{SILVER_VOLUME_PATH}/silver_ride_events")

